RUNTIME: H100 or A100 recommended (T4 fallback)
INSTRUCTIONS:

1. Set runtime to GPU: Runtime → Change runtime type → GPU → H100 (or A100/T4)
2. Run this notebook from inside the cloned repo (e.g., `/content/<repo>/notebooks`) so repo data/utils resolve correctly
3. Ensure required repo data exists in `data/` (test split + any comparison splits needed)
4. Use repo paths (`data/`, `results/`) and save intermediate outputs frequently for crash safety
5. Zero-shot has no training checkpoints, so persist result CSVs after each major run block
6. Save the notebook state regularly (Ctrl+S)


In [4]:
# ── 1. Optional Google Drive mount ────────────────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = "/content/drive/MyDrive/cs4120_project/"
except Exception:
    SAVE_DIR = None

# ── 2. Standard imports ───────────────────────────────────────────
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from sklearn.metrics import f1_score, classification_report
import torch
from transformers import pipeline

# ── 3. Resolve repo/data/results paths ────────────────────────────
repo_candidates = [Path.cwd(), Path.cwd().parent]
repo_root = None
for candidate in repo_candidates:
    if (candidate / "src").exists() and (candidate / "data").exists():
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError("Could not locate repo root with src/ and data/ directories")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

DATA_DIR = repo_root / "data"
RESULTS_DIR = repo_root / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ── 4. Sanity checks ──────────────────────────────────────────────
print(f"PyTorch: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("Repo root:", repo_root)
print("Data dir:", DATA_DIR)
print("Results dir:", RESULTS_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PyTorch: 2.10.0+cu128
GPU available: True
GPU: NVIDIA H100 80GB HBM3
Repo root: /content/cs4120-project
Data dir: /content/cs4120-project/data
Results dir: /content/cs4120-project/results


In [5]:
# ── 5. Data loading helpers ──────────────────────────────────────
import ast
from sklearn.preprocessing import MultiLabelBinarizer
from src.evaluate import evaluate_run

def _parse_labels_column(df, label_col="labels"):
    df = df.copy()
    def _parse_one(value):
        if isinstance(value, list) or isinstance(value, np.ndarray):
            return [int(x) for x in value]
        if isinstance(value, str):
            s = value.strip()
            if s.startswith('[') and s.endswith(']'):
                body = s[1:-1].strip()
                if not body: return []
                toks = [t.strip() for t in body.split(',') if t.strip()] if ',' in body else [t for t in body.split() if t]
                return [int(t) for t in toks]
            return [int(x) for x in ast.literal_eval(s)]
        return value
    df[label_col] = df[label_col].apply(_parse_one)
    return df

def _choose_text_col(df):
    for col in ["text_clean", "text"]:
        if col in df.columns: return col
    raise ValueError("No text column found.")

def _get_label_names(df):
    try:
        ds = load_dataset("google-research-datasets/go_emotions", "simplified")
        return ds["train"].features["labels"].feature.names
    except Exception:
        max_id = max(max(labels) if labels else -1 for labels in df["labels"])
        return [f"label_{i}" for i in range(max_id + 1)]

test_df = _parse_labels_column(pd.read_csv(DATA_DIR / "test.csv"))
text_col = _choose_text_col(test_df)
train_df = _parse_labels_column(pd.read_csv(DATA_DIR / "train.csv")) # Just for label space
label_names = _get_label_names(train_df)

mlb = MultiLabelBinarizer(classes=list(range(len(label_names))))
mlb.fit([list(range(len(label_names)))])
y_test = mlb.transform(test_df["labels"])

print(f"Loaded {len(test_df)} test samples. Labels: {len(label_names)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

simplified/train-00000-of-00001.parquet:   0%|          | 0.00/2.77M [00:00<?, ?B/s]

simplified/validation-00000-of-00001.par(…):   0%|          | 0.00/350k [00:00<?, ?B/s]

simplified/test-00000-of-00001.parquet:   0%|          | 0.00/347k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/43410 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5426 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5427 [00:00<?, ? examples/s]

Loaded 5427 test samples. Labels: 28


In [6]:
# ── 6. Setup Zero-Shot Pipeline ──────────────────────────────────
# Using facebook/bart-large-mnli which is standard for zero-shot text classification
MODEL_NAME = "facebook/bart-large-mnli"
device = 0 if torch.cuda.is_available() else -1

print(f"Loading {MODEL_NAME} on device {device}...")
classifier = pipeline("zero-shot-classification", model=MODEL_NAME, device=device)

Loading facebook/bart-large-mnli on device 0...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [7]:
# ── 7. Run Inference ─────────────────────────────────────────────
# Zero-shot is computationally heavy. Batch processing is recommended.
BATCH_SIZE = 16
texts = test_df[text_col].fillna("").astype(str).tolist()

print("Starting inference (this may take a while)...")
predictions = classifier(
    texts,
    candidate_labels=label_names,
    multi_label=True,
    batch_size=BATCH_SIZE
)

# Convert scores to binary predictions (threshold = 0.5)
THRESHOLD = 0.5
y_pred = np.zeros_like(y_test)

for i, pred in enumerate(predictions):
    for label, score in zip(pred['labels'], pred['scores']):
        if score >= THRESHOLD:
            label_idx = label_names.index(label)
            y_pred[i, label_idx] = 1

print("Inference complete.")

Starting inference (this may take a while)...
Inference complete.


In [8]:
# ── 8. Evaluate and Save Results ─────────────────────────────────
# Since zero-shot uses no training data, data_fraction is conceptually 0.0
# However, for plotting alongside other models, we can record it as "0.0" or "zero_shot"

evaluation = evaluate_run(
    method="zero_shot_bart",
    data_fraction=0.0,
    seed=42, # arbitrary for zero-shot
    label_names=label_names,
    y_true=y_test,
    y_pred=y_pred,
)

overall_df = evaluation["overall"]
per_class_df = evaluation["per_class"]

overall_path = RESULTS_DIR / "zero_shot_overall.csv"
per_class_path = RESULTS_DIR / "zero_shot_per_class.csv"
readme_path = RESULTS_DIR / "zero_shot_results.csv"

overall_df.to_csv(overall_path, index=False)
per_class_df.to_csv(per_class_path, index=False)

readme_df = per_class_df[["method", "data_fraction", "seed", "emotion", "f1", "precision", "recall"]].copy()
readme_df.to_csv(readme_path, index=False)

print("Saved outputs:")
print(overall_path)
print(per_class_path)
display(overall_df)

Saved outputs:
/content/cs4120-project/results/zero_shot_overall.csv
/content/cs4120-project/results/zero_shot_per_class.csv


,method,data_fraction,seed,accuracy,macro_f1,micro_f1,hamming_loss
0,zero_shot_bart,0.0,42,0.007371,0.19028,0.171349,0.199163
